# AI-LUT Training (REQ-004)

Ingests `unified_log.txt`, trains a `DecisionTreeClassifier(max_depth=4)`
predicting ISO from LightLevel, applies the finalized integer ETTR/ALO/HTP
thresholds and the scene WB table, and exports a deterministic `unified.tbl`
plus a `training_audit.log` (with a WB_FALLBACK_COUNT / SKIPPED_ROW_COUNT
summary). Byte-identical output to `lut_training_multi_param.py` for the same
input and the same min-samples setting.


In [ ]:
# Cell 1: Upload unified_log.txt from your SD card (A:/ML/logs/unified_log.txt)
from google.colab import files
uploaded = files.upload()
LOG_PATH = list(uploaded.keys())[0] if uploaded else "unified_log.txt"
print("Using log:", LOG_PATH)


In [ ]:
# Cell 2: Parse the key=value log (entries separated by "---") into a DataFrame
import pandas as pd

def parse_log(path):
    with open(path, "r", encoding="utf-8") as fh:
        raw = fh.read()
    records = []
    for block in raw.split("---"):
        block = block.strip()
        if not block:
            continue
        entry = {}
        for line in block.splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                entry[key.strip()] = value.strip()
        if entry:
            records.append(entry)
    if not records:
        raise ValueError(f"No log entries found in {path!r}; cannot train on an empty log.")
    return pd.DataFrame(records)

df = parse_log(LOG_PATH)
df.head()


In [ ]:
# Cell 3: Feature engineering + audit trail
audit = []
FEATURE_COLUMNS = ["LightLevel"]  # future: add "Shutter", histogram-skew, etc.
if "LightLevel" not in df.columns or "ISO" not in df.columns:
    raise ValueError("Log is missing required 'LightLevel' or 'ISO' column.")
raw_light = df["LightLevel"].copy()
raw_iso = df["ISO"].copy()
df["LightLevel"] = pd.to_numeric(df["LightLevel"], errors="coerce")
df["ISO"] = pd.to_numeric(df["ISO"], errors="coerce")
if "Scene" not in df.columns:
    df["Scene"] = "unknown"
df["Scene"] = df["Scene"].fillna("unknown").astype(str)
bad_mask = df["LightLevel"].isna() | df["ISO"].isna()
for idx in df.index[bad_mask]:
    audit.append(f"SKIPPED_ROW|reason=non-numeric LightLevel/ISO|LightLevel={raw_light.iloc[idx]}|ISO={raw_iso.iloc[idx]}")
df = df[~bad_mask]
df["LightLevel"] = df["LightLevel"].astype(int)
df["ISO"] = df["ISO"].astype(int)
df = df.reset_index(drop=True)
df[["Scene", "LightLevel", "ISO"]]


In [ ]:
# Cell 4: Train DecisionTreeClassifier(max_depth=4) on FEATURE_COLUMNS -> ISO
from sklearn.tree import DecisionTreeClassifier
RANDOM_STATE = 42
RECOMMENDED_MIN_SAMPLES = 200  # <-- collaborators: adjust to your dataset size (advisory only)
n = len(df)
if n < RECOMMENDED_MIN_SAMPLES:
    msg = f"LOW_SAMPLES|count={n}|recommended={RECOMMENDED_MIN_SAMPLES}|note=training proceeds but ISO predictions may be unreliable"
    audit.append(msg)
    print("WARNING:", msg)
clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
clf.fit(df[FEATURE_COLUMNS], df["ISO"])
print("Trained on", n, "samples; classes:", list(clf.classes_))


In [ ]:
# Cell 5: Generate LUT rows using integer threshold rules + scene WB table
WB_TABLE = {
    "daylight": (120, 100, 90),
    "shade": (110, 100, 105),
    "tungsten": (150, 100, 70),
    "lowlight": (130, 100, 80),
    "unknown": (100, 100, 100),
}

def ettr_for(light):
    if light > 200:
        return "reduce_shutter"
    if light < 30:
        return "increase_shutter"
    return "keep_shutter"

def alo_for(light):
    if light < 20:
        return "shadow_boost"
    if light < 50:
        return "shadow_lift"
    return "neutral"

def htp_for(light):
    if light < 30:
        return "priority_on"
    return "priority_off"

def wb_for(scene):
    r, g, b = WB_TABLE.get(scene, (100, 100, 100))
    return f"R{r},G{g},B{b}"

pairs = sorted(set(zip(df["Scene"], df["LightLevel"])))
rows = []
wb_fallback = set()
for scene, light in pairs:
    light = int(light)
    if scene not in WB_TABLE and scene not in wb_fallback:
        wb_fallback.add(scene)
        audit.append(f"WB_FALLBACK_USED|scene={scene}|fallback=R100,G100,B100")
    iso = int(clf.predict(pd.DataFrame({col: [light] for col in FEATURE_COLUMNS}))[0])
    rows.append(f"{scene}|{light}|{ettr_for(light)}|{alo_for(light)}|{htp_for(light)}|{iso}|{wb_for(scene)}")
wb_fallback = sorted(wb_fallback)
rows


In [ ]:
# Cell 6: Write unified.tbl + training_audit.log (UTF-8, no BOM, LF), then download
COLUMN_HEADER = "# Scene|LightLevel|ETTR|ALO|HTP|ISO|WB"
LUT_PATH = "unified.tbl"
AUDIT_PATH = "training_audit.log"
samples = len(df)

header_lines = ["# AI-LUT unified.tbl -- generated by lut_training", f"# training_samples={samples}"]
if samples < RECOMMENDED_MIN_SAMPLES:
    header_lines.append(f"# low_sample_warning: {samples} < recommended {RECOMMENDED_MIN_SAMPLES}")
if wb_fallback:
    header_lines.append("# wb_fallback_scenes=" + ",".join(wb_fallback))
header_lines.append(COLUMN_HEADER)
with open(LUT_PATH, "w", encoding="utf-8", newline="\n") as fh:
    for line in header_lines:
        fh.write(line + "\n")
    for row in rows:
        fh.write(row + "\n")

skipped = sum(1 for line in audit if line.startswith("SKIPPED_ROW"))
summary = [f"TRAINING_SAMPLES={samples}", f"SKIPPED_ROW_COUNT={skipped}", f"WB_FALLBACK_COUNT={len(wb_fallback)}"]
with open(AUDIT_PATH, "w", encoding="utf-8", newline="\n") as fh:
    for line in summary:
        fh.write(line + "\n")
    if audit:
        for line in audit:
            fh.write(line + "\n")
    else:
        fh.write("OK|no detail warnings\n")

print("Wrote", LUT_PATH, "with", len(rows), "rows;", len(audit), "audit note(s)")
print("--- audit ---")
print(chr(10).join(summary + audit))

from google.colab import files
files.download(LUT_PATH)
files.download(AUDIT_PATH)
